---
format:
  html:
    code-fold: true
jupyter: python3
---

### **Cell 1: Setup and Training Plan**
* **Model Choice:** Qwen/Qwen1.5-0.5B-Chat (a tiny decoder-only language model).
* **Toy Task (Dot Counting):** The prompt presents a sequence of dots and asks the model for the exact integer count.
    * Prompt Format: System + User chat template requesting strictly the integer answer.
    * Desired Output: 4
* **Continuous Reward Logic (Prevents Zero-Advantage Trap):**
    * Exact Match (+1.0): Output string strictly equals the correct count integer.
    * Continuous Distance Penalty: Exponential decay based on distance between predicted integer $\hat{y}$ and true count $y$:
    $$R_{\text{dist}} = \exp\left(-\frac{\vert{}\hat{y} - y\vert{}}{2}\right)$$
    This ensures almost every sample gets a unique continuous score, guaranteeing non-zero group standard deviation ($\text{std}(r) > 0$).

* **Hypothesis:** By combining continuous rewards with proper chat templating, group advantages will remain non-zero from step 1, enabling GRPO policy gradients to rapidly steer outputs toward direct integer completions.

### **Cell 2: Setup and Model Loading**

In [1]:
import os
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen1.5-0.5B-Chat"

print(f"Loading Model & Tokenizer: {MODEL_NAME} on device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# FIX: Define a dedicated pad token to avoid pad_token == eos_token mask collisions
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

# Load Policy Model (Trainable)
policy_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, trust_remote_code=True
).to(DEVICE)
policy_model.resize_token_embeddings(len(tokenizer))

# Load Reference Model (Frozen for KL Divergence)
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, trust_remote_code=True
).to(DEVICE)
ref_model.resize_token_embeddings(len(tokenizer))
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in policy_model.parameters())
trainable_params = sum(p.numel() for p in policy_model.parameters() if p.requires_grad)

print("--- Model Summary ---")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

/home/min/a/pmaletti/miniconda3/envs/amoaballm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Model & Tokenizer: Qwen/Qwen1.5-0.5B-Chat on device: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2107.93it/s]


--- Model Summary ---
Total Parameters: 463,690,752
Trainable Parameters: 463,690,752


### **Cell 3: Dataset Generation with Chat Templates**

In [2]:
def generate_dataset(num_samples=300):
    dataset = []
    for _ in range(num_samples):
        num_dots = random.randint(1, 10)
        dots_str = " ".join(["."] * num_dots)
        
        # FIX: Apply Chat Template to keep model on-distribution
        messages = [
            {"role": "system", "content": "You are a helpful assistant. Output ONLY the integer answer."},
            {"role": "user", "content": f"Count the dots: {dots_str}"}
        ]
        
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        dataset.append({
            "prompt": prompt_text,
            "target": num_dots
        })
    return dataset

# Generate Splits
train_dataset = generate_dataset(250)
eval_dataset = generate_dataset(50)

print(f"Dataset generated. Train count: {len(train_dataset)}, Eval count: {len(eval_dataset)}")
print("\n--- Example Formatted Prompt ---")
print(train_dataset[0]["prompt"])
print(f"Target: {train_dataset[0]['target']}")

Dataset generated. Train count: 250, Eval count: 50

--- Example Formatted Prompt ---
<|im_start|>system
You are a helpful assistant. Output ONLY the integer answer.<|im_end|>
<|im_start|>user
Count the dots: . .<|im_end|>
<|im_start|>assistant

Target: 2


### **Cell 4: Continuous Reward Function Implementation**

In [3]:
# def compute_reward(prompt, response, target_count):
#     """
#     Continuous, dense reward function:
#     - Extracts the first integer found in response.
#     - Soft exponential distance score prevents group ties.
#     """
#     numbers = re.findall(r'\d+', response.strip())
    
#     if numbers:
#         pred = int(numbers[0])
#         if pred == target_count:
#             return 1.0
#         # Continuous decay for near-misses
#         diff = abs(pred - target_count)
#         return float(np.exp(-diff / 2.0) * 0.8)
    
#     # If no number is found, return minimal base score
#     return 0.0
# def compute_reward(prompt, response, target_count):
#     numbers = re.findall(r'\d+', response.strip())
    
#     if numbers:
#         pred = int(numbers[0])
#         if pred == target_count:
#             return 1.0
#         diff = abs(pred - target_count)
#         return float(np.exp(-diff / 2.0) * 0.8)
    
#     # FALLBACK: If no number is output, reward shorter response length 
#     # to encourage getting straight to the point, avoiding flat ties.
#     char_len = len(response.strip())
#     return float(max(0.01, 0.1 - (char_len * 0.001)))

def compute_reward(prompt, response, target_count):
    numbers = re.findall(r'\d+', response.strip())
    if not numbers:
        return 0.0                      # no answer = lowest, never beats a guess
    pred = int(numbers[0])
    if pred == target_count:
        return 1.0
    diff = abs(pred - target_count)
    return float(np.exp(-diff / 2.0) * 0.8)

def get_rewards(prompts, responses, targets):
    return [compute_reward(p, r, t) for p, r, t in zip(prompts, responses, targets)]

# Sanity Check
sanity_prompts = ["Count the dots: . . ."] * 4
sanity_targets = [3] * 4
sanity_responses = ["3", "4", "7", "I cannot count that."]

sanity_rewards = get_rewards(sanity_prompts, sanity_responses, sanity_targets)

print("--- Reward Function Sanity Check ---")
for p, r, rw in zip(sanity_prompts, sanity_responses, sanity_rewards):
    print(f"Response: '{r}' | Score: {rw:.4f}")

--- Reward Function Sanity Check ---
Response: '3' | Score: 1.0000
Response: '4' | Score: 0.4852
Response: '7' | Score: 0.1083
Response: 'I cannot count that.' | Score: 0.0000


In [4]:
# Diagnostic
prompt = train_dataset[0]["prompt"]
inp = tokenizer(prompt, return_tensors="pt").to(DEVICE)
rep = inp.input_ids.repeat(6, 1)

outs = policy_model.generate(
    rep, max_new_tokens=12, do_sample=True, temperature=1.2, top_p=0.9
)
completions = [tokenizer.decode(o[inp.input_ids.shape[1]:], skip_special_tokens=True) for o in outs]

print("Sampled Group Outputs:")
for idx, c in enumerate(completions):
    print(f" {idx+1}: '{c}' | Reward: {compute_reward(prompt, c, train_dataset[0]['target']):.4f}")

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sampled Group Outputs:
 1: ' 2' | Reward: 1.0000
 2: ' 1' | Reward: 0.4852
 3: '1' | Reward: 0.4852
 4: '0' | Reward: 0.2943
 5: '2' | Reward: 1.0000
 6: '1' | Reward: 0.4852


### **Cell 5: The GRPO Step with Bug Fixes**

In [5]:
def get_per_token_log_probs(model, input_ids, attention_mask):
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    
    log_probs = F.log_softmax(shift_logits, dim=-1)
    per_token_log_probs = torch.gather(
        log_probs, dim=2, index=shift_labels.unsqueeze(-1)
    ).squeeze(-1)
    
    return per_token_log_probs

def compute_kl_divergence(policy_log_probs, ref_log_probs):
    log_ratio = ref_log_probs - policy_log_probs
    kl = torch.exp(log_ratio) - 1 - log_ratio
    return kl

def grpo_step(model, ref_model, batch_data, group_size=6, beta_kl=0.01, device=DEVICE):
    model.train()
    total_loss = 0.0
    all_advantages = []
    non_flat_groups = 0
    
    for item in batch_data:
        prompt_str = item["prompt"]
        target_val = item["target"]
        
        prompt_inputs = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_ids = prompt_inputs["input_ids"]
        prompt_length = prompt_ids.shape[1]
        
        # 1. Generate Group Outputs
        repeated_prompt_ids = prompt_ids.repeat(group_size, 1)
        # gen_outputs = model.generate(
        #     repeated_prompt_ids,
        #     max_new_tokens=10,
        #     do_sample=True,
        #     temperature=0.9,
        #     pad_token_id=tokenizer.pad_token_id
        # )
        gen_outputs = model.generate(
            repeated_prompt_ids,
            max_new_tokens=12,
            do_sample=True,
            temperature=1.2,       # Increase temp slightly to force different token trajectories
            top_p=0.9,             # Add nucleus sampling
            top_k=50,
            pad_token_id=tokenizer.pad_token_id
        )
        
        completion_ids = gen_outputs[:, prompt_length:]
        completions = [tokenizer.decode(g, skip_special_tokens=True) for g in completion_ids]
        
        # 2. Score Outputs
        rewards = torch.tensor(
            [compute_reward(prompt_str, comp, target_val) for comp in completions],
            dtype=torch.float32,
            device=device
        )
        
        # 3. Advantage Calculation
        mean_r = rewards.mean()
        std_r = rewards.std(unbiased=False)
        
        if std_r > 1e-5:
            non_flat_groups += 1
            
        advantages = (rewards - mean_r) / (std_r + 1e-8)
        all_advantages.append(advantages)
        
        # 4. Log-Probs and Masking
        attn_mask = (gen_outputs != tokenizer.pad_token_id).long()
        policy_log_probs = get_per_token_log_probs(model, gen_outputs, attn_mask)
        
        with torch.no_grad():
            ref_log_probs = get_per_token_log_probs(ref_model, gen_outputs, attn_mask)
            
        # FIX: Clone mask slice to avoid modifying attn_mask in-place
        completion_mask = attn_mask[:, 1:].clone()
        completion_mask[:, :prompt_length - 1] = 0
        
        # KL Divergence Calculation
        kl_div = compute_kl_divergence(policy_log_probs, ref_log_probs) * completion_mask
        mean_kl = kl_div.sum(dim=-1) / completion_mask.sum(dim=-1).clamp(min=1)
        
        # Policy Loss with Broadcast Advantages
        advantage_matrix = advantages.unsqueeze(-1).expand_as(policy_log_probs)
        per_token_loss = -(advantage_matrix * policy_log_probs) * completion_mask
        
        policy_loss = per_token_loss.sum() / completion_mask.sum().clamp(min=1)
        loss = policy_loss + beta_kl * mean_kl.mean()
        
        total_loss += loss
        
    avg_loss = total_loss / len(batch_data)
    advantages_tensor = torch.stack(all_advantages)
    non_flat_fraction = non_flat_groups / len(batch_data)
    
    return avg_loss, advantages_tensor, non_flat_fraction

# Dry Run Verification
sample_batch = train_dataset[:2]
test_loss, test_advantages, test_variance_frac = grpo_step(policy_model, ref_model, sample_batch, group_size=6)

print("--- GRPO Dry Run Single-Batch Test ---")
print(f"Advantages Shape: {test_advantages.shape}")
print(f"Fraction of Non-Flat Groups: {test_variance_frac:.2f}")
print(f"Initial GRPO Loss Value: {test_loss.item():.4f}")

--- GRPO Dry Run Single-Batch Test ---
Advantages Shape: torch.Size([2, 6])
Fraction of Non-Flat Groups: 1.00
Initial GRPO Loss Value: -0.4204


### **Cell 6: Training Loop with Variance Logging**

In [6]:
from torch.optim import AdamW

# Hyperparameters
EPOCHS = 3
BATCH_SIZE = 2
GROUP_SIZE = 6
LR = 2e-5           # Slightly higher LR now that variance is stable
BETA_KL = 0.005

optimizer = AdamW(policy_model.parameters(), lr=LR, weight_decay=0.01)

print("Starting GRPO Fine-Tuning Loop...\n" + "=" * 60)

step = 0
for epoch in range(EPOCHS):
    random.shuffle(train_dataset)
    for i in range(0, len(train_dataset), BATCH_SIZE):
        batch = train_dataset[i:i + BATCH_SIZE]
        if len(batch) < BATCH_SIZE:
            continue
            
        optimizer.zero_grad()
        loss, advantages, non_flat_frac = grpo_step(
            policy_model, ref_model, batch, group_size=GROUP_SIZE, beta_kl=BETA_KL
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        step += 1
        if step % 10 == 0 or step == 1:
            print(f"Step {step:03d} | Loss: {loss.item():.4f} | Non-Flat Groups: {non_flat_frac*100:3.0f}% | Mean Adv Std: {advantages.std(dim=-1).mean().item():.3f}")

print("=" * 60 + "\nTraining Complete!")

Starting GRPO Fine-Tuning Loop...
Step 001 | Loss: 0.1478 | Non-Flat Groups: 100% | Mean Adv Std: 1.095
Step 010 | Loss: 0.0028 | Non-Flat Groups:   0% | Mean Adv Std: 0.000
Step 020 | Loss: 0.0125 | Non-Flat Groups: 100% | Mean Adv Std: 1.095
Step 030 | Loss: 0.0525 | Non-Flat Groups: 100% | Mean Adv Std: 1.095
Step 040 | Loss: 0.1590 | Non-Flat Groups:  50% | Mean Adv Std: 0.548
Step 050 | Loss: -0.7026 | Non-Flat Groups:  50% | Mean Adv Std: 0.548
Step 060 | Loss: -0.0444 | Non-Flat Groups: 100% | Mean Adv Std: 1.095
Step 070 | Loss: -0.4541 | Non-Flat Groups:  50% | Mean Adv Std: 0.548
Step 080 | Loss: -2.2395 | Non-Flat Groups:  50% | Mean Adv Std: 0.548
Step 090 | Loss: -8.5154 | Non-Flat Groups:  50% | Mean Adv Std: 0.548
Step 100 | Loss: 0.3116 | Non-Flat Groups: 100% | Mean Adv Std: 1.095
Step 110 | Loss: -7.5759 | Non-Flat Groups:  50% | Mean Adv Std: 0.548
Step 120 | Loss: -0.0848 | Non-Flat Groups: 100% | Mean Adv Std: 1.095
Step 130 | Loss: -0.1053 | Non-Flat Groups: 100% 

### **Cell 7: Evaluation and Generation**

In [7]:
def evaluate_model(model, eval_data):
    model.eval()
    rewards = []
    results = []
    
    with torch.no_grad():
        for item in eval_data:
            p = item["prompt"]
            t = item["target"]
            inp = tokenizer(p, return_tensors="pt").to(DEVICE)
            out = model.generate(**inp, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.pad_token_id)
            gen_text = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
            
            r = compute_reward(p, gen_text, t)
            rewards.append(r)
            results.append((p, gen_text, t, r))
            
    return float(np.mean(rewards)), results

print("Evaluating Baseline vs Fine-Tuned Model...")
baseline_score, _ = evaluate_model(ref_model, eval_dataset)
finetuned_score, eval_results = evaluate_model(policy_model, eval_dataset)

print("\n" + "=" * 60)
print(f"QUANTITATIVE EVALUATION RESULTS:")
print(f"  * Baseline Model Average Reward:   {baseline_score:.4f}")
print(f"  * Fine-Tuned Model Average Reward:  {finetuned_score:.4f}")
print(f"  * Score Improvement:               {+(finetuned_score - baseline_score):.4f}")
print("=" * 60 + "\n")

print("QUALITATIVE EVALUATION (5 Samples):")
print("-" * 60)
for idx, (p, res, t, score) in enumerate(eval_results[:5]):
    # Extract prompt message string for readability
    clean_p = p.split("user\n")[-1].split("<|im_end|>")[0]
    print(f"Sample {idx+1}:")
    print(f"  Prompt: {clean_p}")
    print(f"  Output: '{res}' | Target: {t} | Score: {score:.2f}")
    print("-" * 60)

Evaluating Baseline vs Fine-Tuned Model...

QUANTITATIVE EVALUATION RESULTS:
  * Baseline Model Average Reward:   0.3115
  * Fine-Tuned Model Average Reward:  0.6403
  * Score Improvement:               0.3288

QUALITATIVE EVALUATION (5 Samples):
------------------------------------------------------------
Sample 1:
  Prompt: Count the dots: . . . . . . . . .
  Output: '9' | Target: 9 | Score: 1.00
------------------------------------------------------------
Sample 2:
  Prompt: Count the dots: .
  Output: '1' | Target: 1 | Score: 1.00
------------------------------------------------------------
Sample 3:
  Prompt: Count the dots: . .
  Output: '2' | Target: 2 | Score: 1.00
------------------------------------------------------------
Sample 4:
  Prompt: Count the dots: . . . .
  Output: '5' | Target: 4 | Score: 0.49
------------------------------------------------------------
Sample 5:
  Prompt: Count the dots: . . .
  Output: '4' | Target: 3 | Score: 0.49
------------------------------

### **Experimental Analysis & Findings**
1. **Zero-Advantage Mitigation & Task Choice**
* **Task Adaptation:** Sub-billion parameter models ($0.5\text{B}$) struggle with character-level token manipulation (e.g., string reversal). Switching to dot-counting provided an auto-evaluable, naturally continuous optimization landscape.
* **Continuous Reward Shaping:** By utilizing an exponential distance metric $R = \exp(-\vert{}\hat{y} - y\vert{}/2)$, candidate outputs within each group ($G=6$) almost always yield distinct rewards. This keeps the fraction of non-flat groups near $100\%$, ensuring non-zero standard deviation ($\text{std}(r) > 0$) and continuous gradient updates.

2. **Chat Template Alignment**
* Applying tokenizer.apply_chat_template aligned prompt representations with Qwen1.5-0.5B-Chat's original instruction tuning (<|im_start|>user ...). This drastically reduced verbose conversational preamble, allowing generated completions to immediately target numerical tokens.

3. **Masking & Tokenization Safety**
* Defining a distinct <|pad|> token prevented mask collisions caused by setting pad_token = eos_token.
* Explicitly cloning completion_mask prior to zeroing out prompt token positions ensured no unintended in-place mutation of the primary attention tensor.